# Style moderne en pandas

Ce notebook définit le **style de code** utilisé dans tout le module pandas.
Il est conçu pour être lu **avant** les notebooks 4, 5, 6, 7 et 8.

L'objectif n'est pas de mémoriser une API : c'est d'apprendre à **lire et valider** du code chaîné,
celui qu'un LLM (Copilot, Claude…) va produire quand vous lui demanderez une transformation pandas.

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
from _data import load_accidents

In [ ]:
df = load_accidents()
df.shape

## 1. Deux styles de code pandas

Regardez les deux cellules ci-dessous. Elles font **exactement la même chose** :
compter le nombre d'accidents par département, sur les accidents survenus entre 6h et 22h,
et afficher le top 10.

Lisez les deux. Observez. On en parle ensuite.

In [ ]:
# --- Style 2015 : impératif, variables intermédiaires ---

df2 = df.copy()
df2["heure"] = df2["hrmn"].str[:2].astype(int)
df2 = df2[df2["heure"] >= 6]
df2 = df2[df2["heure"] <= 22]
grouped = df2.groupby("dep")["Num_Acc"].count()
top10 = grouped.nlargest(10)
top10

In [ ]:
# --- Style 2026 : chaîné, une seule expression ---

(
    df
    .assign(heure=lambda df_: df_["hrmn"].str[:2].astype(int))  # nouvelle colonne
    .query("6 <= heure <= 22")                                    # filtrer
    .groupby("dep")["Num_Acc"]                                    # grouper
    .count()                                                       # agréger
    .nlargest(10)                                                  # top 10
)

**Ce qu'on observe :**

| Style impératif | Style chaîné |
|---|---|
| 6 lignes, 3 variables temporaires | 1 expression, 0 variable intermédiaire |
| `df2` est modifié, `df` aussi potentiellement | `df` n'est jamais modifié |
| Difficile à lire dans l'ordre (il faut chercher `df2`) | Se lit de haut en bas, chaque ligne est une étape |
| Difficile à coller dans une fonction | Facile à envelopper dans une `def` ou un `.pipe()` |

Le style chaîné est aussi ce qu'un LLM produit naturellement quand il génère du pandas moderne.
Savoir le lire — vérifier que chaque étape fait ce qu'on attend — c'est la compétence centrale de ce notebook.

## 2. `.query()` pour filtrer

`.query()` prend une chaîne de caractères qui s'écrit comme une condition SQL/Python.
C'est la méthode par défaut dans ce cours — lisible, chaînable, et un LLM la génère sans effort.

In [ ]:
# Filtrer sur une valeur fixe
df.query("dep == 75").shape

In [ ]:
# Conditions combinées
df.query("dep == 75 and col == 1").shape  # Paris, collision frontale

In [ ]:
# Utiliser une variable Python dans la condition : préfixe @
dep_cible = 69  # Rhône
df.query("dep == @dep_cible").shape

In [ ]:
# Plage de valeurs
df.query("1 <= mois <= 3").shape  # premier trimestre

> **Rappel** : le notebook `3-selectionner_et_filtrer_v2` détaille `.query()` et les alternatives (masques booléens, `.loc[]`). Ce notebook suppose que vous l'avez lu.

## 3. `.assign()` pour ajouter ou modifier une colonne

`.assign()` retourne un **nouveau** DataFrame avec la colonne ajoutée.
Le DataFrame original n'est jamais modifié.

La syntaxe clé : `df.assign(nom_colonne=lambda df_: ...)`

Le `lambda df_: ...` est important — `df_` représente le DataFrame **à l'état courant dans la chaîne**,
pas le `df` original. On va voir pourquoi ça compte.

In [ ]:
# Ajouter une colonne heure extraite de hrmn (format "HH:MM")
(
    df
    .assign(heure=lambda df_: df_["hrmn"].str[:2].astype(int))
    [["Num_Acc", "hrmn", "heure"]]
    .head(5)
)

In [ ]:
# Plusieurs colonnes en une seule fois — les lambdas peuvent se référencer dans l'ordre
(
    df
    .assign(
        heure=lambda df_: df_["hrmn"].str[:2].astype(int),
        tranche=lambda df_: pd.cut(df_["heure"], bins=[0, 6, 12, 18, 24],
                                   labels=["nuit", "matin", "apres-midi", "soir"],
                                   right=False),
    )
    [["hrmn", "heure", "tranche"]]
    .head(8)
)

### Pourquoi le `lambda df_:` et pas juste `df[...]` directement ?

Sans `lambda`, on fait référence à une variable externe — le `df` du contexte courant.
Ça fonctionne... jusqu'à ce qu'on renomme la variable ou qu'on déplace le code.

Voici un scénario concret : on a un pipeline qui tourne, on décide de renommer `df` en `df_accidents`
pour être plus explicite. On fait un chercher-remplacer rapide — mais on en oublie un.

In [ ]:
# Version SANS lambda — fragile au refactoring
# On a renommé df → df_accidents partout... sauf dans le assign

df_accidents = load_accidents()

resultat = (
    df_accidents
    .query("dep == 75")
    .assign(heure=df["hrmn"].str[:2].astype(int))  # BUG : df n'existe plus (ou pire, c'est un autre df)
    .groupby("mois")["Num_Acc"]
    .count()
)
resultat

**Ce qui se passe** : si `df` existe encore dans le notebook (cellule chargée plus haut),
le code tourne sans erreur mais calcule `heure` sur le mauvais objet.
Si `df` n'existe plus, on a un `NameError` — au moins ça plante clairement.

Dans les deux cas, le problème vient du même endroit : le `.assign()` fait référence
à une variable externe au lieu de "ce qui coule dans la chaîne".

In [ ]:
# Version AVEC lambda — robuste
# df_ représente toujours "le DataFrame à cet endroit de la chaîne",
# quelle que soit la variable qui le contient à l'extérieur

resultat = (
    df_accidents
    .query("dep == 75")
    .assign(heure=lambda df_: df_["hrmn"].str[:2].astype(int))
    .groupby("mois")["Num_Acc"]
    .count()
)
resultat

> **Règle** : dans `.assign()`, utilisez toujours `lambda df_: ...`.
> `df_` représente "le DataFrame qui circule dans la chaîne" — pas la variable du contexte.
> Le code reste correct même si vous renommez la variable, déplacez le pipeline dans une fonction,
> ou réordonnez les étapes.

## 4. `.pipe()` pour des étapes nommées

Quand une étape de la chaîne devient complexe (plusieurs lignes, logique non triviale),
on la sort dans une fonction nommée et on la réinsère avec `.pipe()`.

Avantage : la chaîne reste lisible, les étapes sont testables séparément.

In [ ]:
def ajouter_tranche_horaire(df_: pd.DataFrame) -> pd.DataFrame:
    """Extrait l'heure depuis hrmn et ajoute une colonne 'tranche_horaire'."""
    return df_.assign(
        heure=lambda df_: df_["hrmn"].str[:2].astype(int),
        tranche_horaire=lambda df_: pd.cut(
            df_["heure"],
            bins=[0, 6, 12, 18, 24],
            labels=["nuit", "matin", "apres-midi", "soir"],
            right=False,
        ),
    )


def supprimer_accidents_sans_localisation(df_: pd.DataFrame) -> pd.DataFrame:
    """Supprime les lignes sans coordonnées GPS valides."""
    return df_.dropna(subset=["lat", "long"])

In [ ]:
# La chaîne reste propre : chaque .pipe() est une étape nommée
(
    df
    .pipe(ajouter_tranche_horaire)
    .pipe(supprimer_accidents_sans_localisation)
    [["Num_Acc", "hrmn", "heure", "tranche_horaire", "lat", "long"]]
    .head(5)
)

`.pipe(f)` est strictement équivalent à `f(df)` — c'est juste une façon de maintenir le style chaîné
quand l'opération ne peut pas s'écrire comme une méthode pandas.

## 5. Un vrai pipeline sur les accidents

**Question** : parmi les accidents survenus de jour (entre 6h et 22h),
quels sont les 10 départements les plus accidentogènes, et quelle part
représentent les accidents sur voie mouillée ou enneigée (`surf` > 1) ?

In [ ]:
(
    df
    .pipe(ajouter_tranche_horaire)                                           # ajoute heure + tranche_horaire
    .query("tranche_horaire != 'nuit'")                                      # garder uniquement les accidents de jour
    .assign(voie_degradee=lambda df_: (df_["surf"] > 1).astype(int))        # 1 si surface non normale
    .groupby("dep")                                                          # grouper par département
    .agg(
        nb_accidents=("Num_Acc", "count"),                                   # nombre total d'accidents
        nb_voie_degradee=("voie_degradee", "sum"),                           # dont sur surface dégradée
    )
    .assign(pct_voie_degradee=lambda df_: df_["nb_voie_degradee"] / df_["nb_accidents"] * 100)
    .nlargest(10, "nb_accidents")                                            # top 10 par nombre d'accidents
    .round(1)
)

**Comment lire ce pipeline :**

1. `.pipe(ajouter_tranche_horaire)` — on réutilise la fonction définie plus haut
2. `.query(...)` — on filtre sur la tranche horaire, qui n'existait pas dans le `df` original
3. `.assign(voie_degradee=...)` — on crée une colonne indicatrice (0/1)
4. `.groupby("dep").agg(...)` — agrégation nommée (pandas >= 0.25)
5. `.assign(pct_voie_degradee=...)` — calculé sur le résultat du groupby, pas sur `df`
6. `.nlargest(10, ...)` — tri + sélection en une passe

Chaque ligne produit un objet pandas valide. On peut insérer `.head()` n'importe où pour inspecter.

## 6. Ce qu'on ne fait plus

| Ancien code | Remplacement | Pourquoi |
|---|---|---|
| `df.drop_duplicates(inplace=True)` | `df = df.drop_duplicates()` | `inplace=True` est deprecated depuis pandas 2.0 et rend le code difficile à chaîner |
| `df.applymap(f)` | `df.map(f)` | `applymap` renommé `map` en pandas 2.1 |
| `df.append(autre)` | `pd.concat([df, autre])` | `.append()` supprimé en pandas 2.0 |
| `df.ix[...]` | `df.loc[...]` ou `df.iloc[...]` | `.ix` supprimé en pandas 1.0 |
| `df.iteritems()` | `df.items()` | `.iteritems()` supprimé en pandas 2.0 |
| `df[df["a"] > 0]["b"] = 1` | `df.loc[df["a"] > 0, "b"] = 1` | L'indexation chaînée peut modifier une copie sans avertissement |

> Si un LLM vous génère du code avec ces anciens patterns, corrigez-les.
> C'est un signe que le modèle a été entraîné sur du code pandas antérieur à 2020.

## Quiz — lire du code chaîné

Lisez le pipeline suivant **sans l'exécuter**. Répondez aux questions, puis exécutez pour vérifier.

```python
resultat = (
    df
    .assign(heure=lambda df_: df_["hrmn"].str[:2].astype(int))
    .query("lum == 5")          # lum=5 : nuit sans éclairage public
    .query("heure >= 22 or heure <= 5")
    .groupby("dep")["Num_Acc"]
    .count()
    .nlargest(5)
)
```

<details><summary>Question 1 — Que contient <code>resultat</code> ?</summary>

Une **Series** dont l'index est le code département (`dep`) et les valeurs sont le nombre d'accidents
survenus la nuit (entre 22h et 5h) sans éclairage public, pour les 5 départements les plus touchés.

</details>

<details><summary>Question 2 — Pourquoi <code>heure</code> est utilisable dans le deuxième <code>.query()</code> alors qu'elle n'existe pas dans <code>df</code> ?</summary>

Parce que `.assign()` retourne un **nouveau DataFrame** qui contient la colonne `heure`.
Le deuxième `.query()` s'applique à ce nouveau DataFrame, pas à `df`.
C'est la garantie fondamentale du style chaîné : chaque méthode reçoit le résultat de la précédente.

</details>

<details><summary>Question 3 — Quel serait le bug si on écrivait <code>assign(heure=df["hrmn"].str[:2].astype(int))</code> sans lambda ?</summary>

Aucun crash — mais `heure` serait calculée sur le `df` original (toutes les lignes),
puis alignée sur l'index du DataFrame courant. Si une étape précédente avait filtré des lignes,
certaines valeurs ne s'aligneraient plus et deviendraient `NaN`.
Ici il n'y a pas encore de filtre avant le `.assign()`, donc le bug serait invisible — mais le pattern
reste dangereux dès qu'on réordonne les étapes.

</details>

In [ ]:
# Exécutez pour vérifier vos réponses
resultat = (
    df
    .assign(heure=lambda df_: df_["hrmn"].str[:2].astype(int))
    .query("lum == 5")           # nuit sans éclairage public
    .query("heure >= 22 or heure <= 5")
    .groupby("dep")["Num_Acc"]
    .count()
    .nlargest(5)
)
resultat

---

## Pour conclure : le method chaining n'est pas une mode

Le style chaîné n'est pas une préférence stylistique parmi d'autres — c'est la direction que prend l'écosystème.

**Du côté de pandas**, la core team pousse activement vers ce style depuis plusieurs années.
Les nouvelles API (`.assign()`, `.query()`, `.pipe()`, les named aggregations dans `.agg()`) ont toutes été
conçues pour s'insérer dans une chaîne. Tom Augspurger, contributeur historique, a formalisé ce style
dans sa série *Modern Pandas* (2016) — elle reste une référence.

**Du côté de polars**, le method chaining n'est pas une option : c'est *la* seule façon d'écrire du polars.
Il n'existe pas de `inplace=True`, pas de réassignation intermédiaire — l'API l'interdit structurellement.
Si vous savez lire une chaîne pandas, vous pouvez lire du polars le jour où vous en rencontrez.

---

### Faire en sorte qu'un LLM génère du pandas moderne

Un LLM entraîné sur du code public produit souvent du pandas 2015 : `inplace=True`,
`applymap`, boucles `iterrows`, indexation chaînée… parce que c'est ce qu'il a majoritairement vu.

La solution : lui donner des instructions explicites dans un fichier de configuration du projet,
qui sera lu automatiquement à chaque conversation.

**Pour GitHub Copilot** — `.github/copilot-instructions.md` :

```markdown
## Pandas style

- Method chaining only: .query(), .assign(lambda df_: ...), .pipe(), .agg().
- No inplace=True, no applymap (use .map()), no .append() (use pd.concat()), no .ix.
- Prefer .query() for row filtering. For dynamic column selection, use .filter(like="prefix").
- All lambdas in .assign() must use the df_ parameter, never reference an outer variable.
```

**Pour Claude Code** — `CLAUDE.md` à la racine du projet (ce fichier est déjà utilisé dans ce cours) :

```markdown
## Pandas style

- Method chaining only: .query(), .assign(lambda df_: ...), .pipe(), .agg().
- No inplace=True, no applymap (use .map()), no .append() (use pd.concat()), no .ix.
- Prefer .query() for row filtering. For dynamic column selection, use .filter(like="prefix").
- All lambdas in .assign() must use the df_ parameter, never reference an outer variable.
```

Ces instructions fonctionnent aussi pour Cursor (`cursor_rules`), Windsurf, et la plupart des éditeurs
qui permettent de configurer le contexte système du LLM intégré.